# Qwen3.5-35B-A3B BF16 Net-Torque Reciprocal Causal Test

The preceding 900-scene replication found a precise error class in the official non-quantized 35B checkpoint: the signed net torque was recoverable from the outcome-prompt hidden state and could be answered directly, yet the final tipping direction was wrong.

[实验设置与运行说明](../../../docs/experiment_notes/12_cross_study_audits_and_pilots/40ba93e8fb408ca5.md)


In [ ]:
# Notebook utilities: support/visualtorque_checks.py
import base64 as _vt_base64, gzip as _vt_gzip, sys as _vt_sys, types as _vt_types
_vt_source = _vt_gzip.decompress(_vt_base64.b64decode("H4sIAAAAAAACCo1Y647bthL+76dgtziwtLGVTZAGqFs3J8BJkABJESCLoqjXFWiJsnmWElWS2qx7Afo0fbA+SWd4kSnJ23Z/JBLnPvPNcOSLi4vXXDBSHFhxqwltStJIw3ZS3pKS61bQI+kMF9xwpjPyrSS1LJkgUhF23zLFa9YYUrKWNSVrCmS6uLiY8bqVypCqawojpdDh4ED1QfBdeFUsPOmjnlVK1uRgapG1VGmmiKe9uX7/7oM9cSwtNagkkD/A62w2K1lFKgglRxsJ8qSrGYG/ku+ZNmQdjGf6QJ9+8TxJLfUTNwerwolkEiJJ5mo3TwnVINKUgjk9+FdB3MWha24Jbwg3TCWC1ruSrjxnphgtkydXT5+RS4L/pQuym8/Tk4aTR1nXltSwxOpzzihmOtUE+oHduydw1cXnPM8xzDhCLzZ35NWcPBpnYiqf7455Q2uW2JepstjUiQX0AAFyGZFnljw9diZ3R8M0UhMIlj7s8Kg4lvlMBhqpair4z9ZgckdFx0Z+G+WPoRa1vGOtYhW/T3pbqCl/8/LjG3BXsayQdYsxqnny4uvPNi+XP9Dlz1fLL/NtmrxYbeCJLquXy9fbX54/++3X+P3Z1W/AMhQB7fn1q++v8/dv37/6CBaSuWH35jH0EW/mC+LeEOP9S03VbSk/NfMQInpEC5MjNcF/FjY5TK+/lU2IFprs4wHQz5rQpqz0bAsiW8NlQ4U4klvGWt7sCYVY75jSfAfNLqC7u9Y2qoUjGFUMdBQsqakpDhFcbSohDnue7ZXsWt84+KfRBaBars3qydUWKjn/8/c/sKLucPl8te35eeV9JFzjnCEY0bA3eOW1QoN5XhxK7nFjSVvymTc5FI1QYKkD4lDBesTRgx6oMZosUDLd7RKfoAXBimCpCkG1Jvl33Ob0Got1GlRh9EBm85w33OR5opmonHgoaJRn3cEwTdKs5y5kA/UyeXGgChCs16+p0CxKPWjLnDL7GGpvX0CogbaBMGNzC7LZDuVlVWlmkG9zdZ6UgTyM9qTO8N/UDsAaawOtU/GmtDNwfoPYtokZOSiQTgUYwErP+qS4aZlrQ5UxdB9yQ/cLQo1RcWYAEHCOJqGHC8Vbg62jzVGw8VwdmQSxiUWIYmBvami9Huj5Wwtng8LR5Q1EIy9g+0HVDnH9kSthCUYG8wA1DiqextqDEPQHMg4NCN4AfAspurohPsg9M63UUUfbILEqgcPjYIPSZEmeYIc7HdPEeNhltMV1IEmsooXX94gIuFltRhbB0XQ883Ay/t3Me8foHbMQ4bvOIKTBr1uEvUUGPiAw3MzgNd0zmwhAjwH1/cTz+8V62L6D1oz4soqxMnF9H50WQmrmU4dt4aOFyBd+bNo2wakL4kHIpSgCBer1jbpZWR3bMDzhf3sMKv0Q9WMJT0eZC9fIg9n7nLyDVAFQteENxQtCg7oa7iXSaQo5WIC/hehKvC5sVomRVs8S94SSgGe8AlM6C1mwkwMGAdw/3OANerO9STY/ptvLm/TXgzGtfrF6/Hjz443++pvtozTMiMEKMM/+L3mD6TGIYE7+Q54SBqNuiPvW5nYM+egPK8AXxOlpoApdDT1mmJXUE6DZ/LpFYUFq2GLXw4t6mj+7y0LWoo5EtKLTKI9zI7rdhyGE4kyZ+9t/GvPMNzUgtAFcNEXvr+DaRAj6nDS7CvciQ+ASqTmUhZK6E7i2Q9PCRoQlhWLu8EUqrCV2CFRU0WPW68FC2IkTiuL2qOiS1KAU6CEViZOY1sUX1wk4eKArOsF1BNCs19cKNGN0XilMLG/e5uF0O3tVvUmfAm/xfH4g4jTWM+oUKICfo9Mq9zGWvHDT9tTgtnKArWjDG8z2QHc6xtMdzzbIso0Q5GAYEx0ahykdpPO0evzPLX4fDXx01M5cfsd1RwUU+acOr1ekgDnM9uyhjcRxxbsIzvKTsHs4iX9ScH1F28wDkpnjm2yz6QQmeC/4rWpgw0HGGbLP6fBDzKIb8m1pZ+5p5wBSI82V6OCjCOmRtrD+xe47xjROG9yWePX0mcO5ONXiuZJYm+f1SMxhoYNiHjxRJ5dU7eHuury8/YRPYeJAjCjmdx9Tys7vPiVTKl5++lr1to86mIyXD8cW7d7hg/8k6N2dn0MSGHdb6DjZI6OLETa91n4I4/chbEC5/3JJYP4xBcBn5iDL3GmAPHT424K3JRXfw7UlohjPSdlODoOzl4GAbbCQz17Y0yaRep/OhOoXNPv+3/6XDUAZbXWvL+3h0nY7mNOHB2rrXcSTYTLxBL8bB9MKD2FDh8XJats8WW1PhYU+qfp0IVacoaFadxbmmnflDMcmKIqnlPVhTJ+0cUjBJGKXMp+P7Gy6w4yK4XSuvl5JAJId/EL0QIKRKcRoZXxtsTb9VUs24viVuyDAWfzFwm2NxYGLctkqWTAYskJC1sDIEW4gIfoF0n6lgq0ArWidU8dh5u1PVm8/HCGKJvxkBSjMeWuPTmW8L1hryFvL8Uopqc59HzhURh/g6MQ6Vpik/+DhRNFkHDkN4y616jL/lkMloHd8PXAq4V05/xeSBygCsNvhnLu1xX4yoY7wChid92EMxhNqemA65aj5TN8667CHN7C6KraH5Qm/WVvFctU1eQFUkBonIR3JP2gxhq5PrX3/C0ov0eJaFQAA"))
_vt_checks = _vt_types.ModuleType("visualtorque_checks")
exec(compile(_vt_source, "<visualtorque_checks>", "exec"), _vt_checks.__dict__)
_vt_sys.modules["visualtorque_checks"] = _vt_checks
_vt_checks.install_display()
del _vt_source


In [ ]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 17 tracked cells.')


## 1. Frozen errors, strict controls, and causal protocol


In [ ]:
%%cell_progress 1/17 Mount Drive and locate VisualTorque
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'toolkit' / 'pyproject.toml').is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


In [ ]:
%%cell_progress 2/17 Install and verify the multimodal environment
import importlib
import json
import os
import subprocess
import sys

cell_stage('Configuring the 96GB CUDA process before importing Torch')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)

cell_stage('Installing pinned inference dependencies without replacing NumPy')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    '--upgrade-strategy', 'only-if-needed',
    'transformers==5.14.0', 'accelerate>=1.10',
    'sentencepiece', 'protobuf', 'huggingface_hub>=0.34',
    'tqdm>=4.66',
])

cell_stage('Checking the preinstalled numerical stack in a fresh process')
probe_code = '''
import json
import matplotlib
import numpy
import pandas
import scipy
import sklearn
print(json.dumps({
    "numpy": numpy.__version__, "scipy": scipy.__version__,
    "pandas": pandas.__version__, "matplotlib": matplotlib.__version__,
    "sklearn": sklearn.__version__,
}, sort_keys=True))
'''
probe = subprocess.run(
    [sys.executable, '-c', probe_code],
    text=True, capture_output=True, check=False,
)
if probe.returncode != 0:
    raise RuntimeError(
        'The numerical environment is already inconsistent. Restart the '
        'session and run this notebook from the first cell; do not upgrade '
        'NumPy/Pandas/SciPy inside the live process.\n'
        + probe.stderr[-3000:]
    )

cell_stage('Registering the live VisualTorque source tree')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
assert (PACKAGE_ROOT / '__init__.py').is_file(), PACKAGE_ROOT
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if (
        module_name == 'visual_torque_toolkit'
        or module_name.startswith('visual_torque_toolkit.')
    ):
        del sys.modules[module_name]
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
import torch
import torchvision
import visual_torque_toolkit
from IPython.display import display
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter

torch.backends.cuda.matmul.allow_tf32 = True
required_features = (
    '_template', '_to_device', '_candidate_plan', '_append_text_prefix',
)
assert all(
    hasattr(HuggingFaceMultimodalAdapter, name)
    for name in required_features
)
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents, toolkit_path
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('Analysis stack:', probe.stdout.strip().splitlines()[-1])
print('Toolkit source:', toolkit_path)


In [ ]:
%%cell_progress 3/17 Verify the 96GB GPU and Hugging Face access
cell_stage('Checking CUDA capacity')
assert torch.cuda.is_available(), 'Select a CUDA runtime with at least 90 GiB.'
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
print('GPU:', GPU_NAME)
print('CUDA memory (GiB):', round(GPU_MEMORY_GB, 1))
assert GPU_MEMORY_GB >= 90, (
    'The original BF16 checkpoint requires one GPU with at least 90 GiB.',
    GPU_NAME, GPU_MEMORY_GB,
)

cell_stage('Checking local checkpoint space')
import shutil
LOCAL_DISK_FREE_GB = shutil.disk_usage('/content').free / 2**30
print('Local disk free (GiB):', round(LOCAL_DISK_FREE_GB, 1))
if LOCAL_DISK_FREE_GB < 85:
    print(
        'Warning: the official BF16 repository is about 72 GB. Free more '
        'local disk or point the Hugging Face cache to a larger volume.'
    )

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. The public checkpoint may still work.')
except Exception as exc:
    print('Hugging Face login skipped:', exc)


In [ ]:
%%cell_progress 4/17 Lock the reciprocal net-torque protocol
cell_stage('Freezing parent evidence, strict pairs, sites, strengths, and decisions')
import hashlib
import json
from datetime import datetime, timezone

MODEL_ID = 'Qwen/Qwen3.5-35B-A3B'
MODEL_SLUG = 'Qwen__Qwen3.5-35B-A3B__671d2e01'
MODEL_REVISION = '671d2e016ab7fd44882530043d29818c1440af2d'
MODEL_DTYPE = 'bfloat16'
DEVICE_MAP = 'auto'
EXPECTED_LAYER_COUNT = 40
NET_LAYER = 20
SEMANTIC_LAYER = 30
LABEL_LAYER = 34
CAPTURE_LAYERS = (NET_LAYER, SEMANTIC_LAYER, LABEL_LAYER)
SITE_SCOPE = 'final_prompt_token'

SEED = 20261218
DIRECTION_SEED = 20260901
LABELS = ('A', 'B', 'C')
MAPPING_VARIANTS = ('forward', 'reverse')
DONOR_TASKS = ('weight_more', 'weight_less')
QUESTION_KINDS = (
    'outcome', 'left_units', 'right_units', 'left_distance',
    'right_distance', 'left_torque', 'right_torque', 'net_torque',
)
ALPHAS = (0.5, 1.0, 2.0)
CONDITIONS = (
    'l20_net_exact', 'l20_net_reverse', 'l20_net_random',
    'l30_semantic_exact', 'l30_semantic_reverse', 'l30_semantic_random',
    'l34_code_exact', 'l34_code_stale', 'l34_code_random',
)
PRIMARY_ALPHA = 1.0
MINIMUM_CANDIDATE_ERRORS = 12
MINIMUM_ACTIVE_PAIRS = 8
MAXIMUM_PAIRS = 24
MAX_MATCH_DISTANCE = 8.0
PATCH_CONDITION_BATCH_SIZE = 1
BOOTSTRAP_REPLICATES = 10000
ALPHA_LEVEL = 0.05
MINIMUM_STAGE_EFFECT = 0.20
SMOKE_CENTERED_LOGPROB_TOLERANCE = 0.15
NORM_RELATIVE_TOLERANCE = 0.02

AUDIT_ROOT = (
    PROJECT_ROOT / 'results'
    / 'qwen35b_a3b_bf16_fracture_replication_v1'
)
AUDIT_RUN_PATH = AUDIT_ROOT / 'RUN_COMPLETE.json'
AUDIT_PROTOCOL_PATH = AUDIT_ROOT / 'frozen_protocol.json'
AUDIT_OUTCOME_MANIFEST_PATH = AUDIT_ROOT / 'outcome_prompt_manifest.jsonl'
AUDIT_CONFIRMATION_PATH = AUDIT_ROOT / 'confirmation_scene_manifest.jsonl'
AUDIT_PROBE_LOCK_PATH = AUDIT_ROOT / 'frozen_probe_selection.json'
AUDIT_FRACTURE_PATH = AUDIT_ROOT / 'analysis' / 'scene_fracture_map.csv'
AUDIT_OUTCOME_SCORES_PATH = AUDIT_ROOT / 'analysis' / 'all_outcome_scores.csv'
AUDIT_OUTCOME_SHARD_DIR = AUDIT_ROOT / 'outcome_capture_shards'

DIRECTION_ROOT = PROJECT_ROOT / 'results' / 'semantic_label_depth_localization_v1'
DIRECTION_SUPPORT_PATH = DIRECTION_ROOT / 'depth_support.jsonl'
INVERSE_SCENE_DIR = (
    PROJECT_ROOT / 'results' / 'inverse_query_weight_route_specificity_v1'
    / 'rendered_scenes'
)

OUTPUT_ROOT = (
    PROJECT_ROOT / 'results'
    / 'qwen35b_a3b_bf16_nettorque_reciprocal_causal_v1'
)
PROTOCOL_LOCK_PATH = OUTPUT_ROOT / 'frozen_protocol.json'
PAIR_MANIFEST_PATH = OUTPUT_ROOT / 'strict_pair_manifest.jsonl'
PAIR_LOCK_PATH = OUTPUT_ROOT / 'strict_pair_lock.json'
REPLAY_PATH = OUTPUT_ROOT / 'mapping_replay.jsonl'
ACTIVE_PAIR_PATH = OUTPUT_ROOT / 'active_pair_manifest.jsonl'
ACTIVE_LOCK_PATH = OUTPUT_ROOT / 'active_pair_lock.json'
SHARD_DIR = OUTPUT_ROOT / 'pair_shards'
ANALYSIS_DIR = OUTPUT_ROOT / 'analysis'
FIGURE_DIR = ANALYSIS_DIR / 'figures'
for current in (OUTPUT_ROOT, SHARD_DIR, ANALYSIS_DIR, FIGURE_DIR):
    current.mkdir(parents=True, exist_ok=True)

from visualtorque_checks import sha256_file_by_name as sha256_file

def stable_key(value, seed=SEED):
    return hashlib.sha256(f'{seed}:{value!r}'.encode('utf-8')).hexdigest()

def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if torch.is_tensor(value) and value.numel() == 1:
        return value.item()
    raise TypeError(type(value).__name__)

def write_json_atomic(file_path, value):
    file_path = Path(file_path)
    temporary = file_path.with_suffix(file_path.suffix + '.tmp')
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False,
                   default=json_default) + '\n', encoding='utf-8'
    )
    temporary.replace(file_path)

def write_jsonl_atomic(file_path, rows):
    file_path = Path(file_path)
    temporary = file_path.with_suffix(file_path.suffix + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(
                row, sort_keys=True, default=json_default
            ) + '\n')
    temporary.replace(file_path)

def read_jsonl(file_path):
    with Path(file_path).open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

required_inputs = (
    AUDIT_RUN_PATH, AUDIT_PROTOCOL_PATH, AUDIT_OUTCOME_MANIFEST_PATH,
    AUDIT_CONFIRMATION_PATH, AUDIT_PROBE_LOCK_PATH,
    AUDIT_FRACTURE_PATH, AUDIT_OUTCOME_SCORES_PATH,
    DIRECTION_SUPPORT_PATH, INVERSE_SCENE_DIR,
)
for required in required_inputs:
    assert Path(required).exists(), f'Missing prerequisite: {required}'
parent_run = json.loads(AUDIT_RUN_PATH.read_text(encoding='utf-8'))
assert parent_run['model_id'] == MODEL_ID
assert parent_run['revision'] == MODEL_REVISION
assert parent_run['next_experiment_authorized'] is True

protocol_identity = {
    'schema_version': 'qwen35b-nettorque-reciprocal-causal-1.0',
    'analysis_status': 'confirmatory_on_preexisting_probe_test_errors',
    'model_id': MODEL_ID, 'revision': MODEL_REVISION,
    'parent_run_sha256': sha256_file(AUDIT_RUN_PATH),
    'parent_fracture_sha256': sha256_file(AUDIT_FRACTURE_PATH),
    'sites': {
        'net_torque': [NET_LAYER, SITE_SCOPE],
        'physical_semantic': [SEMANTIC_LAYER, SITE_SCOPE],
        'response_code': [LABEL_LAYER, SITE_SCOPE],
    },
    'site_selection': (
        'L20 frozen by 35B probe_train/probe_dev; L30/L34 are normalized-depth '
        'transfers fixed before target interventions'
    ),
    'target_rule': (
        'preexisting probe_test net_torque_to_outcome errors with correct '
        'direct net-torque answer and held-out net-torque-sign readout'
    ),
    'pair_rule': (
        'unique correct probe_test control with identical exact outcome, '
        'diagnostic cell, weight cue, and distance cue; bounded physical distance'
    ),
    'mapping_rule': (
        'error stays on the same wrong physical side and control stays correct '
        'under two mappings that swap exact and wrong answer letters'
    ),
    'conditions': list(CONDITIONS), 'alphas': list(ALPHAS),
    'primary_alpha': PRIMARY_ALPHA,
    'primary_evidence': (
        'paired reciprocal evidence: correct-direction repair in errors and '
        'reversed-direction induction in matched correct scenes'
    ),
    'decision_order': [
        'L20 passes -> insufficient net-torque strength is causally sufficient',
        'L30 passes -> net-torque-to-physical-semantic handoff fracture',
        'L34 passes -> physical-semantic-to-response-code fracture',
        'otherwise unresolved at frozen sites',
    ],
    'bootstrap_unit': 'strict matched physical pair',
    'seed': SEED,
}
if PROTOCOL_LOCK_PATH.is_file():
    previous = json.loads(PROTOCOL_LOCK_PATH.read_text(encoding='utf-8'))
    assert previous.get('identity') == protocol_identity, (
        'Existing protocol differs. Use a new versioned output directory.'
    )
else:
    write_json_atomic(PROTOCOL_LOCK_PATH, {
        'identity': protocol_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })
print('Model:', MODEL_ID, '@', MODEL_REVISION)
print('Frozen causal sites:', CAPTURE_LAYERS)
print('Output:', OUTPUT_ROOT)


In [ ]:
%%cell_progress 5/17 Prepare Dataset v1 on local SSD
cell_stage('Checking local and Drive dataset caches')
from zipfile import ZipFile

REQUIRED_DATASET_FILES = {
    'dataset_config.json', 'scenes.jsonl', 'counterfactual_pairs.jsonl',
    'questions.jsonl', 'splits.json', 'statistics.json',
}
DATASET_DIR = Path('/content/visualtorque_dataset_v1')
DATASET_ARCHIVE = PROJECT_ROOT / 'data' / 'dataset_v1.zip'
LEGACY_DATASET_DIR = PROJECT_ROOT / 'data' / 'dataset_v1'

def dataset_complete(path):
    return path.is_dir() and all(
        (path / name).is_file() for name in REQUIRED_DATASET_FILES
    )

if not dataset_complete(DATASET_DIR):
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    if DATASET_ARCHIVE.is_file():
        cell_stage('Extracting Dataset v1 archive')
        with ZipFile(DATASET_ARCHIVE, 'r') as bundle:
            members = [m for m in bundle.infolist() if not m.is_dir()]
            root = DATASET_DIR.resolve()
            for member in tqdm(members, desc='Extracting Dataset v1', unit='file'):
                target = (DATASET_DIR / member.filename).resolve()
                if root not in target.parents:
                    raise ValueError(f'Unsafe archive path: {member.filename}')
                bundle.extract(member, DATASET_DIR)
    elif dataset_complete(LEGACY_DATASET_DIR):
        DATASET_DIR = LEGACY_DATASET_DIR
    else:
        raise FileNotFoundError(
            'Run the Dataset v1 build/cache cell in the behavior notebook first.'
        )
assert dataset_complete(DATASET_DIR)
print('Dataset:', DATASET_DIR)


In [ ]:
%%cell_progress 6/17 Build strict error-correct physical pairs
cell_stage('Loading the frozen fracture class and matching without interventions')
from collections import Counter, defaultdict
import shutil

from visual_torque_toolkit.behavior import (
    BehaviorRunConfig, PromptCondition, load_behavior_examples,
)

scene_records = read_jsonl(DATASET_DIR / 'scenes.jsonl')

def is_single_stack_scene(row):
    stacks = row.get('stacks', [])
    counts = Counter(str(stack.get('side')) for stack in stacks)
    return bool(
        str(row.get('metadata', {}).get('family', '')) == 'single_stack'
        and str(row.get('scene_id', '')).startswith('single_')
        and len(stacks) == 2 and counts == {'left': 1, 'right': 1}
    )

single_scenes = [row for row in scene_records if is_single_stack_scene(row)]
assert len(single_scenes) == 900
scene_by_id = {str(row['scene_id']): row for row in single_scenes}

def sign(value):
    return int(value > 0) - int(value < 0)

def scene_features(scene):
    left = next(stack for stack in scene['stacks'] if stack['side'] == 'left')
    right = next(stack for stack in scene['stacks'] if stack['side'] == 'right')
    derived = scene['derived']
    left_units, right_units = int(left['units']), int(right['units'])
    left_distance, right_distance = int(left['distance']), int(right['distance'])
    left_torque = int(derived['left_torque'])
    right_torque = int(derived['right_torque'])
    net_torque = int(derived['net_torque'])
    torque_sign = sign(net_torque)
    weight_cue = sign(left_units - right_units)
    distance_cue = sign(left_distance - right_distance)
    if torque_sign and weight_cue != torque_sign and distance_cue == torque_sign:
        diagnostic_cell = 'weight_conflict'
    elif torque_sign and distance_cue != torque_sign and weight_cue == torque_sign:
        diagnostic_cell = 'distance_conflict'
    elif torque_sign and weight_cue == torque_sign and distance_cue == torque_sign:
        diagnostic_cell = 'both_cues_aligned'
    else:
        diagnostic_cell = 'mixed_or_balanced'
    return {
        'left_units': left_units, 'right_units': right_units,
        'left_distance': left_distance, 'right_distance': right_distance,
        'left_torque': left_torque, 'right_torque': right_torque,
        'net_torque': net_torque, 'outcome': str(derived['outcome']),
        'weight_cue': weight_cue, 'distance_cue': distance_cue,
        'torque_sign': torque_sign, 'diagnostic_cell': diagnostic_cell,
        'margin_bin': min(abs(net_torque) // 4, 5),
    }

physical_by_scene = {
    scene_id: scene_features(scene) for scene_id, scene in scene_by_id.items()
}

loader_config = BehaviorRunConfig(
    dataset_dir=DATASET_DIR, output_dir=OUTPUT_ROOT / '_unused_loader',
    model_id=MODEL_ID, model_revision=MODEL_REVISION,
    conditions=(PromptCondition.IMAGE_RULE,), question_kinds=QUESTION_KINDS,
    splits=('probe_train', 'probe_dev', 'probe_test'), batch_size=1,
    generation_every=0, seed=SEED,
)
all_examples = tuple(load_behavior_examples(loader_config))
examples_by_scene_kind = defaultdict(list)
for example in all_examples:
    scene_id = str(example.scene_id)
    if scene_id in scene_by_id:
        examples_by_scene_kind[(scene_id, example.question_kind)].append(example)

audit_manifest = read_jsonl(AUDIT_OUTCOME_MANIFEST_PATH)
outcome_example_by_scene = {}
for row in audit_manifest:
    scene_id = str(row['scene_id'])
    outcome_example_by_scene[scene_id] = next(
        example for example in examples_by_scene_kind[(scene_id, 'outcome')]
        if str(example.variant) == str(row['variant'])
    )

fracture_frame = pd.read_csv(AUDIT_FRACTURE_PATH)
fracture_frame['scene_id'] = fracture_frame['scene_id'].astype(str)
for column in ('net_torque_task_pass', 'net_torque_probe_pass'):
    if fracture_frame[column].dtype != bool:
        fracture_frame[column] = (
            fracture_frame[column].astype(str).str.lower() == 'true'
        )
behavior_frame = pd.read_csv(AUDIT_OUTCOME_SCORES_PATH)
behavior_frame['scene_id'] = behavior_frame['scene_id'].astype(str)
if behavior_frame['correct'].dtype != bool:
    behavior_frame['correct'] = (
        behavior_frame['correct'].astype(str).str.lower() == 'true'
    )
behavior_by_scene = behavior_frame.set_index('scene_id')

candidate_fractures = fracture_frame[
    (fracture_frame['cohort'] == 'outcome_error')
    & (fracture_frame['candidate_fracture'] == 'net_torque_to_outcome')
    & fracture_frame['net_torque_task_pass'].astype(bool)
    & fracture_frame['net_torque_probe_pass'].astype(bool)
].copy()
candidate_rows = []
for row in candidate_fractures.to_dict('records'):
    scene_id = str(row['scene_id'])
    features = physical_by_scene[scene_id]
    exact = str(features['outcome'])
    wrong = str(row['outcome_prediction'])
    if exact in ('left', 'right') and wrong in ('left', 'right') and exact != wrong:
        candidate_rows.append({
            'scene_id': scene_id, 'exact_semantic': exact,
            'shortcut_semantic': wrong,
            'outcome_confidence_gap': float(row['outcome_confidence_gap']),
            **features,
        })
assert len(candidate_rows) >= MINIMUM_CANDIDATE_ERRORS, len(candidate_rows)

correct_pool = []
for scene_id, row in behavior_by_scene.iterrows():
    if str(row['split']) != 'probe_test' or not bool(row['correct']):
        continue
    features = physical_by_scene[scene_id]
    if features['outcome'] not in ('left', 'right'):
        continue
    correct_pool.append({'scene_id': scene_id, **features})

def match_distance(error, control):
    return float(
        abs(int(error['margin_bin']) - int(control['margin_bin']))
        + 0.25 * abs(int(error['left_units']) - int(control['left_units']))
        + 0.25 * abs(int(error['right_units']) - int(control['right_units']))
        + 0.15 * abs(int(error['left_distance']) - int(control['left_distance']))
        + 0.15 * abs(int(error['right_distance']) - int(control['right_distance']))
        + 0.10 * abs(int(error['net_torque']) - int(control['net_torque']))
    )

def eligible_controls(error, used):
    values = []
    for control in correct_pool:
        if control['scene_id'] in used:
            continue
        if any(
            control[key] != error[key]
            for key in ('outcome', 'diagnostic_cell', 'weight_cue', 'distance_cue')
        ):
            continue
        if abs(int(control['margin_bin']) - int(error['margin_bin'])) > 1:
            continue
        distance = match_distance(error, control)
        if distance <= MAX_MATCH_DISTANCE:
            values.append((distance, stable_key(
                ('match', error['scene_id'], control['scene_id'])
            ), control))
    return sorted(values, key=lambda item: (item[0], item[1]))

candidate_rows.sort(key=lambda row: (
    len(eligible_controls(row, set())), stable_key(('candidate', row['scene_id']))
))
used_controls = set()
strict_pairs = []
for error in candidate_rows:
    choices = eligible_controls(error, used_controls)
    if not choices:
        continue
    distance, _, control = choices[0]
    used_controls.add(control['scene_id'])
    strict_pairs.append({
        'pair_id': 'pair_' + stable_key((
            'strict', error['scene_id'], control['scene_id']
        ))[:20],
        'error_scene_id': error['scene_id'],
        'control_scene_id': control['scene_id'],
        'exact_semantic': error['exact_semantic'],
        'shortcut_semantic': error['shortcut_semantic'],
        'diagnostic_cell': error['diagnostic_cell'],
        'weight_cue': int(error['weight_cue']),
        'distance_cue': int(error['distance_cue']),
        'error_net_torque': int(error['net_torque']),
        'control_net_torque': int(control['net_torque']),
        'match_distance': float(distance),
    })
    if len(strict_pairs) == MAXIMUM_PAIRS:
        break
assert len(strict_pairs) >= MINIMUM_ACTIVE_PAIRS, {
    'strict_pairs': len(strict_pairs), 'candidate_errors': len(candidate_rows),
    'maximum_match_distance': MAX_MATCH_DISTANCE,
}
strict_pairs.sort(key=lambda row: row['pair_id'])
write_jsonl_atomic(PAIR_MANIFEST_PATH, strict_pairs)
pair_identity = {
    'schema_version': 'strict-nettorque-error-correct-pairs-1.0',
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'selection_uses_current_interventions': False,
    'pair_ids': [row['pair_id'] for row in strict_pairs],
}
if PAIR_LOCK_PATH.is_file():
    previous = json.loads(PAIR_LOCK_PATH.read_text(encoding='utf-8'))
    assert previous.get('identity') == pair_identity
else:
    write_json_atomic(PAIR_LOCK_PATH, {
        'identity': pair_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })

direction_support = read_jsonl(DIRECTION_SUPPORT_PATH)
EXPECTED_DIRECTION_FAMILIES = len(direction_support)
DIRECTION_RUNTIME_DIR = Path('/content/visualtorque_35b_direction_images')
DIRECTION_RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
direction_files = sorted({
    role['image_filename']
    for row in direction_support for role in row['roles'].values()
})
for filename in tqdm(direction_files, desc='Caching direction images', unit='image'):
    source = INVERSE_SCENE_DIR / filename
    destination = DIRECTION_RUNTIME_DIR / filename
    assert source.is_file(), source
    if not destination.is_file() or destination.stat().st_size != source.stat().st_size:
        shutil.copy2(source, destination)

pair_manifest = strict_pairs
print('Frozen net-to-outcome candidates:', len(candidate_rows))
print('Strict physical pairs:', len(pair_manifest))
print('Maximum retained match distance:', max(row['match_distance'] for row in pair_manifest))
print('Pair cells:', Counter(row['diagnostic_cell'] for row in pair_manifest))


## 2. Model-specific directions and mapping-stable pairs


In [ ]:
%%cell_progress 7/17 Load pinned Qwen35B-A3B BF16 and causal layers
cell_stage('Loading the official non-quantized BF16 checkpoint')
import gc
import itertools

if 'adapter' in globals():
    del adapter
    gc.collect()
    torch.cuda.empty_cache()

adapter = HuggingFaceMultimodalAdapter(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=MODEL_DTYPE,
    device_map=DEVICE_MAP,
)
adapter.model.config.use_cache = False
if hasattr(adapter.model, 'generation_config'):
    adapter.model.generation_config.use_cache = False
display(adapter.metadata)

quantization_config = getattr(adapter.model.config, 'quantization_config', None)
assert quantization_config is None, (
    'Expected the original non-quantized checkpoint.', quantization_config
)
device_map = getattr(adapter.model, 'hf_device_map', {}) or {}
offloaded = {
    name: placement for name, placement in device_map.items()
    if str(placement).lower() in {'cpu', 'disk'}
}
assert not offloaded, (
    'The 96GB protocol does not allow CPU/disk offload.', offloaded
)
parameter_storage_gib = sum(
    parameter.numel() * parameter.element_size()
    for parameter in adapter.model.parameters()
) / 2**30
assert parameter_storage_gib > 55, (
    'Loaded storage is too small for the original BF16 checkpoint.',
    parameter_storage_gib,
)

def resolve_module_path(root, path):
    value = root
    for component in path.split('.'):
        value = getattr(value, component)
    return value

def discover_decoder_layers(model):
    candidates = (
        'model.language_model.layers', 'model.text_model.layers',
        'language_model.model.layers', 'model.model.layers', 'model.layers',
    )
    for path in candidates:
        try:
            layers = resolve_module_path(model, path)
        except AttributeError:
            continue
        if isinstance(layers, torch.nn.ModuleList) and len(layers) >= 8:
            return path, layers
    raise RuntimeError('Could not locate decoder layers.')

LAYER_PATH, DECODER_LAYERS = discover_decoder_layers(adapter.model)
assert len(DECODER_LAYERS) == EXPECTED_LAYER_COUNT
print('Layer path:', LAYER_PATH)
print('Decoder layers:', len(DECODER_LAYERS))
print('Parameter storage (GiB):', round(parameter_storage_gib, 2))
print('CUDA allocated after load (GiB):', round(
    torch.cuda.memory_allocated() / 2**30, 2
))
print('CUDA reserved after load (GiB):', round(
    torch.cuda.memory_reserved() / 2**30, 2
))


In [ ]:
%%cell_progress 8/17 Define counterbalanced target and direction prompts
cell_stage('Separating physical sides from A/B/C response codes')
import math
from dataclasses import replace as dataclass_replace
from visual_torque_toolkit.behavior.data import BehaviorExample

RULE_TEXT = (
    'Each square is one unit of weight. Distance is measured in tick '
    'intervals from the center pivot. Torque equals weight times distance. '
    'Total torque on a side is the sum over stacks on that side.'
)
QUESTION_TEXT = {
    'outcome': 'Which way will the balance scale tip?',
    'weight_more': 'Which side has more total weight?',
    'weight_less': 'Which side has less total weight?',
}
SEMANTIC_TEXT = {'left': 'left side', 'balanced': 'equal', 'right': 'right side'}

def outcome(value):
    return 'left' if value > 0 else 'right' if value < 0 else 'balanced'

def task_semantic(features, task):
    if task == 'outcome':
        value = int(features['net_torque'])
    elif task == 'weight_more':
        value = int(features['weight_score'])
    elif task == 'weight_less':
        value = -int(features['weight_score'])
    else:
        raise KeyError(task)
    return outcome(value)

def mapping_by_label(semantics):
    assert set(semantics) == {'left', 'balanced', 'right'}
    return tuple({
        'position': index, 'label': LABELS[index], 'semantic': semantic,
    } for index, semantic in enumerate(semantics))

def direction_reference_mapping(row):
    semantics = ['left', 'balanced', 'right']
    offset = int(stable_key(
        f"options:{row['family_id']}", seed=DIRECTION_SEED
    )[:8], 16) % 3
    return mapping_by_label(semantics[offset:] + semantics[:offset])

def target_mappings(row):
    exact, wrong = row['exact_semantic'], row['shortcut_semantic']
    assert {exact, wrong} == {'left', 'right'}
    balanced_label = LABELS[int(stable_key(
        ('balanced-label', row['scene_id'])
    )[:8], 16) % 3]
    side_labels = [label for label in LABELS if label != balanced_label]
    result = {}
    for variant, exact_label, wrong_label in (
        ('forward', side_labels[0], side_labels[1]),
        ('reverse', side_labels[1], side_labels[0]),
    ):
        by_label = {
            exact_label: exact, wrong_label: wrong, balanced_label: 'balanced'
        }
        result[variant] = mapping_by_label(tuple(by_label[label] for label in LABELS))
    return result

def semantics_by_label(mapping):
    return tuple(next(
        item['semantic'] for item in mapping if item['label'] == label
    ) for label in LABELS)

def label_for(mapping, semantic):
    return next(item['label'] for item in mapping if item['semantic'] == semantic)

def custom_prompt(task, mapping):
    options = '\n'.join(
        f"{item['label']}. {SEMANTIC_TEXT[item['semantic']]}" for item in mapping
    )
    return '\n\n'.join([
        'Answer this question using only the supplied balance-scale scene.',
        RULE_TEXT, 'Question: ' + QUESTION_TEXT[task], 'Options:\n' + options,
        'Do not explain your reasoning. Reply with exactly one option letter '
        'and nothing else. Valid replies are: A, B, C.',
    ])

def make_direction_example(row, role_record, task, mapping, name):
    features = role_record['features']
    semantics = semantics_by_label(mapping)
    gold = task_semantic(features, task)
    return BehaviorExample(
        example_id=f"fracture_direction:{row['family_id']}:{name}",
        question_id=f"fracture_direction_{row['family_id']}_{name}",
        scene_id=str(role_record['scene_id']), condition=PromptCondition.IMAGE_RULE,
        question_kind=task, variant='fracture_direction', split='direction_support',
        problem_type='semantic_direction', outcome=gold,
        net_torque=int(features['net_torque']), prompt=custom_prompt(task, mapping),
        image_path=(DIRECTION_RUNTIME_DIR / role_record['image_filename']).resolve(),
        labels=LABELS, option_text=tuple(SEMANTIC_TEXT[x] for x in semantics),
        option_semantics=semantics, gold_label=LABELS[semantics.index(gold)],
        gold_semantic=gold,
    )

def build_direction_examples(row):
    mapping = direction_reference_mapping(row)
    examples = {}
    for task in DONOR_TASKS:
        for role in ('base', 'aligned_source'):
            key = f'{task}__{role}'
            examples[key] = make_direction_example(
                row, row['roles'][role], task, mapping, key
            )
    return examples, mapping

def build_target_examples(row):
    mappings = target_mappings(row)
    base = outcome_example_by_scene[row['scene_id']]
    examples = {}
    for variant, mapping in mappings.items():
        semantics = semantics_by_label(mapping)
        examples[variant] = dataclass_replace(
            base,
            example_id=f"fracture_target:{row['scene_id']}:{variant}",
            question_id=f"fracture_target_{row['scene_id']}_{variant}",
            variant=f'fracture_{variant}', prompt=custom_prompt('outcome', mapping),
            labels=LABELS, option_text=tuple(SEMANTIC_TEXT[x] for x in semantics),
            option_semantics=semantics,
            gold_label=LABELS[semantics.index(row['exact_semantic'])],
            gold_semantic=row['exact_semantic'],
        )
    return examples, mappings

def output_hidden_tensor(output):
    if torch.is_tensor(output):
        hidden = output
    elif isinstance(output, (tuple, list)) and output and torch.is_tensor(output[0]):
        hidden = output[0]
    elif hasattr(output, 'last_hidden_state') and torch.is_tensor(output.last_hidden_state):
        hidden = output.last_hidden_state
    else:
        raise TypeError(f'Unsupported output: {type(output).__name__}')
    assert hidden.ndim == 3
    return hidden

def replace_output_hidden(output, modified):
    if torch.is_tensor(output):
        return modified
    if isinstance(output, tuple):
        return (modified,) + output[1:]
    if isinstance(output, list):
        return [modified] + output[1:]
    raise TypeError(type(output).__name__)

def prepare_examples(examples):
    examples = tuple(examples)
    plan = adapter._candidate_plan(examples[0])
    if plan is None:
        raise RuntimeError('A/B/C labels must be single-token candidates.')
    for example in examples[1:]:
        assert adapter._candidate_plan(example) == plan
    fixed_prefix, token_by_label = plan
    inputs = adapter._append_text_prefix(
        adapter._to_device(adapter._template(
            [example.messages() for example in examples], add_generation_prompt=True,
        )), fixed_prefix,
    )
    return {'examples': examples, 'inputs': inputs, 'token_by_label': token_by_label}

def final_mask(prepared):
    assert len(prepared['examples']) == 1
    attention = prepared['inputs']['attention_mask'][0].detach().cpu().bool()
    mask = torch.zeros_like(attention)
    mask[int(torch.where(attention)[0][-1])] = True
    return mask

print('Prompt builders ready.')


In [ ]:
%%cell_progress 9/17 Define final-token capture and intervention scorers
cell_stage('Defining variable-layer final-token transport')

def score_records(prepared, logits):
    logprobs = torch.log_softmax(logits[:, -1, :].float(), dim=-1)
    records = []
    for index, example in enumerate(prepared['examples']):
        scores = [
            float(logprobs[index, prepared['token_by_label'][label]].detach().cpu())
            for label in example.labels
        ]
        maximum = max(scores)
        weights = [math.exp(value - maximum) for value in scores]
        probabilities = [value / sum(weights) for value in weights]
        predicted = int(np.argmax(scores))
        records.append({
            'candidate_labels': list(example.labels),
            'candidate_semantics': list(example.option_semantics),
            'candidate_logprobs': scores,
            'candidate_probabilities': probabilities,
            'predicted_label': example.labels[predicted],
            'predicted_semantic': example.option_semantics[predicted],
            'gold_label': example.gold_label,
            'gold_semantic': example.gold_semantic,
        })
    return records

def forward_capture(prepared, layers):
    hidden_by_layer = {}
    handles = []
    for layer in layers:
        def hook(_module, _inputs, output, layer=layer):
            hidden_by_layer[layer] = output_hidden_tensor(
                output
            ).detach().cpu().clone()
        handles.append(DECODER_LAYERS[layer].register_forward_hook(hook))
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        for handle in handles:
            handle.remove()
    return score_records(prepared, output.logits), hidden_by_layer

def capture_examples(examples, layers):
    scores = {}
    hidden = {name: {} for name in examples}
    masks = {}
    for name, example in examples.items():
        prepared = prepare_examples([example])
        masks[name] = final_mask(prepared)
        current_scores, current_hidden = forward_capture(prepared, layers)
        scores[name] = current_scores[0]
        for layer, tensor in current_hidden.items():
            hidden[name][layer] = tensor[0]
        del prepared, current_hidden
    return scores, hidden, masks

def selected_norm(tensor):
    return float(torch.linalg.vector_norm(tensor.float()))

def matched_random(tensor, key):
    generator = torch.Generator(device='cpu')
    generator.manual_seed(int(stable_key(key)[:16], 16) % (2**63 - 1))
    noise = torch.randn(tensor.shape, generator=generator, dtype=torch.float32)
    return (noise * (selected_norm(tensor) / selected_norm(noise))).to(tensor.dtype)

def forward_intervention_chunk(prepared, conditions):
    assert len(prepared['examples']) == len(conditions)
    layers = {int(row['layer_index']) for row in conditions}
    assert len(layers) == 1
    layer = next(iter(layers))
    payloads = torch.stack([row['payload_selected'] for row in conditions])
    masks = torch.stack([row['target_mask'] for row in conditions])
    modes = [row['intervention_mode'] for row in conditions]
    counts = []
    def hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        payload = payloads.to(device=hidden.device, dtype=hidden.dtype)
        mask = masks.to(device=hidden.device)
        modified = hidden.clone()
        counts.clear()
        for index, mode in enumerate(modes):
            current = mask[index]
            assert int(current.sum()) == int(payload[index].shape[0]) == 1
            if mode == 'add':
                modified[index, current] = hidden[index, current] + payload[index]
            elif mode == 'replace':
                modified[index, current] = payload[index]
            else:
                raise ValueError(mode)
            counts.append(int(current.sum()))
        return replace_output_hidden(output, modified)
    handle = DECODER_LAYERS[layer].register_forward_hook(hook)
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        handle.remove()
    records = score_records(prepared, output.logits)
    for record, condition, count in zip(records, conditions, counts):
        record.update({
            key: value for key, value in condition.items()
            if key not in ('payload_selected', 'target_mask')
        })
        record['patched_tokens'] = count
    return records

def score_conditions(example, conditions):
    records = []
    grouped = defaultdict(list)
    for condition in conditions:
        grouped[int(condition['layer_index'])].append(condition)
    for layer_conditions in grouped.values():
        for start in range(0, len(layer_conditions), PATCH_CONDITION_BATCH_SIZE):
            actual = layer_conditions[start:start + PATCH_CONDITION_BATCH_SIZE]
            padded = list(actual)
            while len(padded) < PATCH_CONDITION_BATCH_SIZE:
                padded.append(padded[-1])
            prepared = prepare_examples([example] * len(padded))
            records.extend(forward_intervention_chunk(
                prepared, padded
            )[:len(actual)])
            del prepared
    return records

def centered_scores(record):
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    return values - values.mean()

def label_score(record, label):
    labels = list(record['candidate_labels'])
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    index = labels.index(label)
    other = np.delete(values, index)
    return float(values[index] - np.logaddexp.reduce(other))

def entropy(record):
    probabilities = np.asarray(record['candidate_probabilities'], dtype=float)
    return float(-np.sum(probabilities * np.log(np.clip(probabilities, 1e-12, 1))))

def cosine_similarity(left, right):
    left = left.float().reshape(-1)
    right = right.float().reshape(-1)
    denominator = torch.linalg.vector_norm(left) * torch.linalg.vector_norm(right)
    assert float(denominator) > 0
    return float(torch.dot(left, right) / denominator)

def rescale_direction(tensor, target_norm):
    norm = selected_norm(tensor)
    assert norm > 0 and target_norm > 0
    return (tensor.float() * (float(target_norm) / norm)).to(tensor.dtype)

def unit_direction(tensor):
    return rescale_direction(tensor, 1.0).float()

def score_l49_and_capture_l55(example, payload_selected, clean_l55, metadata):
    prepared = prepare_examples([example])
    mask = final_mask(prepared)
    captured = {}
    def semantic_hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        payload = payload_selected.to(device=hidden.device, dtype=hidden.dtype)
        current = mask.to(hidden.device)
        modified = hidden.clone()
        modified[0, current] = hidden[0, current] + payload
        return replace_output_hidden(output, modified)
    def label_capture_hook(_module, _inputs, output):
        captured['hidden'] = output_hidden_tensor(output).detach().cpu().clone()
    semantic_handle = DECODER_LAYERS[SEMANTIC_LAYER].register_forward_hook(
        semantic_hook
    )
    label_handle = DECODER_LAYERS[LABEL_LAYER].register_forward_hook(
        label_capture_hook
    )
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        semantic_handle.remove()
        label_handle.remove()
    record = score_records(prepared, output.logits)[0]
    downstream = captured['hidden'][0][mask] - clean_l55
    record.update(metadata)
    record['patched_tokens'] = int(mask.sum())
    record['downstream_l55_norm'] = selected_norm(downstream)
    return record, downstream

print('Semantic steering and downstream capture scorers ready.')


In [ ]:
%%cell_progress 10/17 Build model-specific physical and answer directions
cell_stage('Learning directions without any target intervention output')
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

left_units, left_raw_norms = [], []
code_units, code_raw_norms = defaultdict(list), []
progress = tqdm(direction_support, desc='L30/L34 direction library', unit='family')
for row in progress:
    examples, mapping = build_direction_examples(row)
    scores, hidden, masks = capture_examples(examples, CAPTURE_LAYERS)
    for task in DONOR_TASKS:
        base_key, source_key = f'{task}__base', f'{task}__aligned_source'
        base_semantic = task_semantic(row['roles']['base']['features'], task)
        source_semantic = task_semantic(
            row['roles']['aligned_source']['features'], task
        )
        assert {base_semantic, source_semantic} == {'left', 'right'}
        semantic_delta = (
            hidden[source_key][SEMANTIC_LAYER][masks[source_key]]
            - hidden[base_key][SEMANTIC_LAYER][masks[base_key]]
        )
        left_delta = (
            semantic_delta if source_semantic == 'left' else -semantic_delta
        )
        left_units.append(unit_direction(left_delta))
        left_raw_norms.append(selected_norm(semantic_delta))
        code_delta = (
            hidden[source_key][LABEL_LAYER][masks[source_key]]
            - hidden[base_key][LABEL_LAYER][masks[base_key]]
        )
        transition = (
            label_for(mapping, base_semantic),
            label_for(mapping, source_semantic),
        )
        code_units[transition].append(unit_direction(code_delta))
        code_raw_norms.append(selected_norm(code_delta))
    del hidden
    torch.cuda.empty_cache()

assert len(left_units) == EXPECTED_DIRECTION_FAMILIES * len(DONOR_TASKS)
assert set(code_units) == {
    (left, right) for left in LABELS for right in LABELS if left != right
}
SEMANTIC_STEERING_NORM = float(np.median(left_raw_norms))
CODE_STEERING_NORM = float(np.median(code_raw_norms))
SEMANTIC_DIRECTION = rescale_direction(
    torch.stack(left_units).mean(dim=0), SEMANTIC_STEERING_NORM
)
CODE_DIRECTIONS = {
    transition: rescale_direction(
        torch.stack(values).mean(dim=0), CODE_STEERING_NORM
    ) for transition, values in code_units.items()
}

audit_hidden_rows, audit_scene_ids = [], []
for shard_path in tqdm(
    sorted(AUDIT_OUTCOME_SHARD_DIR.glob('outcome_*.pt')),
    desc='Loading frozen L20 outcome states', unit='shard',
):
    payload = torch.load(shard_path, map_location='cpu', weights_only=False)
    assert str(NET_LAYER) in payload['hidden']
    audit_hidden_rows.append(payload['hidden'][str(NET_LAYER)].float())
    audit_scene_ids.extend(str(value) for value in payload['scene_ids'])
X = torch.cat(audit_hidden_rows, dim=0).numpy()
audit_metadata = pd.DataFrame(audit_manifest)
assert audit_metadata['scene_id'].astype(str).tolist() == audit_scene_ids
y = audit_metadata['net_torque'].to_numpy(dtype=float)
fit = audit_metadata['split'].isin(['probe_train', 'probe_dev']).to_numpy()
probe_lock = json.loads(AUDIT_PROBE_LOCK_PATH.read_text(encoding='utf-8'))
net_selection = probe_lock['identity']['selection']['net_torque']
assert int(net_selection['layer']) == NET_LAYER, net_selection
scaler = StandardScaler().fit(X[fit])
ridge = Ridge(alpha=float(net_selection['alpha']), solver='lsqr').fit(
    scaler.transform(X[fit]), y[fit]
)
effective = ridge.coef_ / scaler.scale_
net_unit = unit_direction(torch.from_numpy(
    effective.astype('float32')
).reshape(1, -1))
projection = X[fit] @ net_unit.reshape(-1).numpy()
positive, negative = projection[y[fit] > 0], projection[y[fit] < 0]
NET_STEERING_NORM = float(
    abs(np.median(positive) - np.median(negative)) / 2
)
assert NET_STEERING_NORM > 0
NET_DIRECTION = rescale_direction(net_unit, NET_STEERING_NORM)

direction_metadata = {
    'schema_version': 'qwen35b-reciprocal-direction-library-1.0',
    'net_layer': NET_LAYER, 'semantic_layer': SEMANTIC_LAYER,
    'label_layer': LABEL_LAYER,
    'net_direction': 'left-minus-right',
    'semantic_direction': 'left-minus-right',
    'net_norm': NET_STEERING_NORM,
    'semantic_norm': SEMANTIC_STEERING_NORM,
    'code_norm': CODE_STEERING_NORM,
    'net_probe_lock_sha256': sha256_file(AUDIT_PROBE_LOCK_PATH),
    'direction_support_sha256': sha256_file(DIRECTION_SUPPORT_PATH),
}
write_json_atomic(OUTPUT_ROOT / 'direction_library.json', direction_metadata)
print(direction_metadata)


In [ ]:
%%cell_progress 11/17 Freeze mapping-stable reciprocal pairs
cell_stage('Replaying errors and controls under swapped A/B/C mappings')

def pair_role_row(pair, role):
    return {
        'scene_id': pair[f'{role}_scene_id'],
        'exact_semantic': pair['exact_semantic'],
        'shortcut_semantic': pair['shortcut_semantic'],
    }

replay_rows = []
for pair in tqdm(pair_manifest, desc='Two-mapping reciprocal replay', unit='pair'):
    for role in ('error', 'control'):
        role_row = pair_role_row(pair, role)
        examples, mappings = build_target_examples(role_row)
        scores, _, _ = capture_examples(examples, ())
        for variant in MAPPING_VARIANTS:
            expected = (
                pair['shortcut_semantic'] if role == 'error'
                else pair['exact_semantic']
            )
            replay_rows.append({
                'pair_id': pair['pair_id'], 'role': role,
                'scene_id': role_row['scene_id'],
                'mapping_variant': variant,
                'predicted_semantic': scores[variant]['predicted_semantic'],
                'expected_baseline': expected,
                'stable': bool(scores[variant]['predicted_semantic'] == expected),
            })
write_jsonl_atomic(REPLAY_PATH, replay_rows)
replay_frame = pd.DataFrame(replay_rows)
stability = replay_frame.groupby(['pair_id', 'role'])['stable'].all().unstack()
active_ids = set(stability[
    stability.get('error', False) & stability.get('control', False)
].index)
active_pair_manifest = [
    pair for pair in pair_manifest if pair['pair_id'] in active_ids
]
assert len(active_pair_manifest) >= MINIMUM_ACTIVE_PAIRS, {
    'strict_pairs': len(pair_manifest),
    'mapping_stable_reciprocal_pairs': len(active_pair_manifest),
    'minimum': MINIMUM_ACTIVE_PAIRS,
}
write_jsonl_atomic(ACTIVE_PAIR_PATH, active_pair_manifest)
active_identity = {
    'schema_version': 'mapping-stable-reciprocal-pairs-1.0',
    'pair_lock_sha256': sha256_file(PAIR_LOCK_PATH),
    'selection_uses_intervention_outputs': False,
    'pair_ids': [row['pair_id'] for row in active_pair_manifest],
}
if ACTIVE_LOCK_PATH.is_file():
    previous = json.loads(ACTIVE_LOCK_PATH.read_text(encoding='utf-8'))
    assert previous.get('identity') == active_identity
else:
    write_json_atomic(ACTIVE_LOCK_PATH, {
        'identity': active_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })
print('Strict / mapping-stable reciprocal pairs:',
      len(pair_manifest), len(active_pair_manifest))
print('Active cells:', Counter(
    row['diagnostic_cell'] for row in active_pair_manifest
))


In [ ]:
%%cell_progress 12/17 Run parity, mapping, and norm smoke tests
cell_stage('Checking exact replay and equal-strength controls')

smoke_pair = active_pair_manifest[0]
smoke_row = pair_role_row(smoke_pair, 'error')
smoke_examples, smoke_mappings = build_target_examples(smoke_row)
smoke_scores, smoke_hidden, smoke_masks = capture_examples(
    smoke_examples, CAPTURE_LAYERS
)
self_errors = {}
for layer in CAPTURE_LAYERS:
    for variant in MAPPING_VARIANTS:
        record = score_conditions(smoke_examples[variant], [{
            'condition': 'self_replay', 'layer_index': layer,
            'intervention_mode': 'replace',
            'payload_selected': smoke_hidden[variant][layer][smoke_masks[variant]],
            'target_mask': smoke_masks[variant],
        }])[0]
        error = float(np.max(np.abs(
            centered_scores(record) - centered_scores(smoke_scores[variant])
        )))
        assert error < SMOKE_CENTERED_LOGPROB_TOLERANCE, (
            layer, variant, error
        )
        self_errors[f'{layer}:{variant}'] = error

for name, tensor in (
    ('net', NET_DIRECTION),
    ('semantic', SEMANTIC_DIRECTION),
    ('code', next(iter(CODE_DIRECTIONS.values()))),
):
    random = matched_random(tensor, ('smoke', name))
    relative = abs(selected_norm(random) - selected_norm(tensor)) / selected_norm(tensor)
    assert relative < NORM_RELATIVE_TOLERANCE, (name, relative)

for variant in MAPPING_VARIANTS:
    mapping = smoke_mappings[variant]
    other = smoke_mappings[
        'reverse' if variant == 'forward' else 'forward'
    ]
    assert label_for(mapping, smoke_pair['exact_semantic']) == label_for(
        other, smoke_pair['shortcut_semantic']
    )
control_row = pair_role_row(smoke_pair, 'control')
control_examples, _ = build_target_examples(control_row)
control_scores, _, _ = capture_examples(control_examples, ())
assert all(
    control_scores[variant]['predicted_semantic'] == smoke_pair['exact_semantic']
    for variant in MAPPING_VARIANTS
)
print('Maximum centered self-replay error:', max(self_errors.values()))


## 3. Reciprocal repair, induction, and causal decision


In [ ]:
%%cell_progress 13/17 Run reciprocal repair and induction
cell_stage('Repairing errors and reversing the same routes in correct controls')

def semantic_margin(record, exact_semantic, wrong_semantic):
    semantics = list(record['candidate_semantics'])
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    return float(
        values[semantics.index(exact_semantic)]
        - values[semantics.index(wrong_semantic)]
    )

capture_identity = {
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'active_lock_sha256': sha256_file(ACTIVE_LOCK_PATH),
    'direction_library_sha256': sha256_file(
        OUTPUT_ROOT / 'direction_library.json'
    ),
    'conditions': list(CONDITIONS), 'alphas': list(ALPHAS),
}
CAPTURE_SHA256 = 'sha256:' + hashlib.sha256(
    json.dumps(capture_identity, sort_keys=True).encode('utf-8')
).hexdigest()

progress = tqdm(
    active_pair_manifest, desc='Reciprocal causal pairs', unit='pair'
)
for pair in progress:
    shard_path = SHARD_DIR / f"{pair['pair_id']}.json"
    if shard_path.is_file():
        existing = json.loads(shard_path.read_text(encoding='utf-8'))
        assert existing['capture_identity_sha256'] == CAPTURE_SHA256
        continue
    pair_records = []
    for role in ('error', 'control'):
        role_row = pair_role_row(pair, role)
        examples, mappings = build_target_examples(role_row)
        clean_scores, _, clean_masks = capture_examples(examples, ())
        for variant in MAPPING_VARIANTS:
            example, mapping = examples[variant], mappings[variant]
            clean = clean_scores[variant]
            target_mask = clean_masks[variant]
            exact, wrong = pair['exact_semantic'], pair['shortcut_semantic']
            exact_label = label_for(mapping, exact)
            wrong_label = label_for(mapping, wrong)
            other_variant = 'reverse' if variant == 'forward' else 'forward'
            other_mapping = mappings[other_variant]
            stale_transition = (
                label_for(other_mapping, wrong),
                label_for(other_mapping, exact),
            )
            sign_value = 1.0 if exact == 'left' else -1.0
            base_payloads = {
                'l20_net_exact': sign_value * NET_DIRECTION,
                'l20_net_reverse': -sign_value * NET_DIRECTION,
                'l20_net_random': matched_random(
                    NET_DIRECTION, (pair['pair_id'], role, variant, 'l20')
                ),
                'l30_semantic_exact': sign_value * SEMANTIC_DIRECTION,
                'l30_semantic_reverse': -sign_value * SEMANTIC_DIRECTION,
                'l30_semantic_random': matched_random(
                    SEMANTIC_DIRECTION, (pair['pair_id'], role, variant, 'l30')
                ),
                'l34_code_exact': CODE_DIRECTIONS[(wrong_label, exact_label)],
                'l34_code_stale': CODE_DIRECTIONS[stale_transition],
                'l34_code_random': matched_random(
                    CODE_DIRECTIONS[(wrong_label, exact_label)],
                    (pair['pair_id'], role, variant, 'l34'),
                ),
            }
            layer_by_condition = {
                condition: (
                    NET_LAYER if condition.startswith('l20_')
                    else SEMANTIC_LAYER if condition.startswith('l30_')
                    else LABEL_LAYER
                ) for condition in CONDITIONS
            }
            clean_margin = semantic_margin(clean, exact, wrong)
            conditions = []
            for alpha in ALPHAS:
                for condition in CONDITIONS:
                    conditions.append({
                        'pair_id': pair['pair_id'], 'role': role,
                        'scene_id': role_row['scene_id'],
                        'mapping_variant': variant,
                        'diagnostic_cell': pair['diagnostic_cell'],
                        'condition': condition, 'alpha': float(alpha),
                        'layer_index': layer_by_condition[condition],
                        'intervention_mode': 'add',
                        'payload_selected': base_payloads[condition] * float(alpha),
                        'target_mask': target_mask,
                    })
            current_records = score_conditions(example, conditions)
            for record in current_records:
                margin = semantic_margin(record, exact, wrong)
                record.update({
                    'exact_semantic': exact, 'shortcut_semantic': wrong,
                    'exact_label': exact_label, 'shortcut_label': wrong_label,
                    'clean_predicted_semantic': clean['predicted_semantic'],
                    'clean_margin': clean_margin,
                    'intervention_margin': margin,
                    'margin_shift': margin - clean_margin,
                    'predicted_exact': bool(
                        record['predicted_semantic'] == exact
                    ),
                    'predicted_wrong': bool(
                        record['predicted_semantic'] == wrong
                    ),
                    'payload_norm': selected_norm(
                        base_payloads[record['condition']]
                        * float(record['alpha'])
                    ),
                })
            pair_records.extend(current_records)
    expected = 2 * len(MAPPING_VARIANTS) * len(ALPHAS) * len(CONDITIONS)
    assert len(pair_records) == expected
    write_json_atomic(shard_path, {
        'schema_version': 'qwen35b-reciprocal-causal-pair-shard-1.0',
        'capture_identity_sha256': CAPTURE_SHA256,
        'pair_id': pair['pair_id'], 'records': pair_records,
    })
    progress.set_postfix_str(pair['diagnostic_cell'])
    torch.cuda.empty_cache()
print('Completed pair shards:', len(list(SHARD_DIR.glob('*.json'))))


In [ ]:
%%cell_progress 14/17 Assemble paired causal estimands
cell_stage('Averaging answer mappings within each physical scene')
import gc

all_records = []
for pair in active_pair_manifest:
    document = json.loads(
        (SHARD_DIR / f"{pair['pair_id']}.json").read_text(encoding='utf-8')
    )
    assert document['capture_identity_sha256'] == CAPTURE_SHA256
    all_records.extend(document['records'])
record_frame = pd.DataFrame(all_records)
record_frame.to_csv(ANALYSIS_DIR / 'reciprocal_intervention_records.csv', index=False)

estimand_frame = record_frame.groupby(
    ['pair_id', 'role', 'diagnostic_cell', 'condition', 'alpha'],
    as_index=False,
).agg(
    margin_shift=('margin_shift', 'mean'),
    predicted_exact=('predicted_exact', 'mean'),
    predicted_wrong=('predicted_wrong', 'mean'),
    clean_margin=('clean_margin', 'mean'),
    intervention_margin=('intervention_margin', 'mean'),
    payload_norm=('payload_norm', 'mean'),
)
assert estimand_frame.groupby(
    ['pair_id', 'role', 'condition', 'alpha']
).size().eq(1).all()
estimand_frame.to_csv(ANALYSIS_DIR / 'mapping_averaged_estimands.csv', index=False)

del smoke_hidden
if 'adapter' in globals():
    del adapter
gc.collect()
torch.cuda.empty_cache()
print('Active physical pairs:', estimand_frame['pair_id'].nunique())
display(estimand_frame.groupby(['role', 'condition', 'alpha'])[
    ['margin_shift', 'predicted_exact', 'predicted_wrong']
].mean().round(4))


In [ ]:
%%cell_progress 15/17 Test the reciprocal causal decision tree
cell_stage('Bootstrapping strict matched physical pairs')

def paired_interval(values, key):
    values = np.asarray(values, dtype=float)
    generator = np.random.default_rng(
        int(stable_key(('bootstrap', key))[:16], 16) % (2**63 - 1)
    )
    indices = generator.integers(
        0, len(values), size=(BOOTSTRAP_REPLICATES, len(values))
    )
    draws = values[indices].mean(axis=1)
    return {
        'estimate': float(values.mean()),
        'ci_low': float(np.quantile(draws, ALPHA_LEVEL / 2)),
        'ci_high': float(np.quantile(draws, 1 - ALPHA_LEVEL / 2)),
        'pairs': int(len(values)),
    }

primary = estimand_frame[
    np.isclose(estimand_frame['alpha'], PRIMARY_ALPHA)
].copy()
margin_pivot = primary.pivot(
    index='pair_id', columns=['role', 'condition'], values='margin_shift'
)
exact_pivot = primary.pivot(
    index='pair_id', columns=['role', 'condition'], values='predicted_exact'
)
wrong_pivot = primary.pivot(
    index='pair_id', columns=['role', 'condition'], values='predicted_wrong'
)

stage_specs = {
    'net_torque_strength': (
        'l20_net_exact', 'l20_net_reverse', 'l20_net_random'
    ),
    'physical_semantic_handoff': (
        'l30_semantic_exact', 'l30_semantic_reverse', 'l30_semantic_random'
    ),
    'response_code_handoff': (
        'l34_code_exact', 'l34_code_stale', 'l34_code_random'
    ),
}
test_rows = []
stage_decisions = {}
for stage, (exact_condition, negative_condition, random_condition) in stage_specs.items():
    quantities = {
        'error_exact_shift': margin_pivot[('error', exact_condition)],
        'error_exact_minus_random': (
            margin_pivot[('error', exact_condition)]
            - margin_pivot[('error', random_condition)]
        ),
        'error_exact_minus_reverse': (
            margin_pivot[('error', exact_condition)]
            - margin_pivot[('error', negative_condition)]
        ),
        'control_induction_vs_random': (
            margin_pivot[('control', random_condition)]
            - margin_pivot[('control', negative_condition)]
        ),
        'control_exact_minus_induction': (
            margin_pivot[('control', exact_condition)]
            - margin_pivot[('control', negative_condition)]
        ),
        'reciprocal_score': 0.5 * (
            margin_pivot[('error', exact_condition)]
            - margin_pivot[('error', random_condition)]
            + margin_pivot[('control', random_condition)]
            - margin_pivot[('control', negative_condition)]
        ),
    }
    results = {}
    for metric, values in quantities.items():
        result = paired_interval(values.to_numpy(), (stage, metric))
        results[metric] = result
        test_rows.append({'stage': stage, 'metric': metric, **result})
    error_rescue_rate = float(exact_pivot[('error', exact_condition)].mean())
    induction_rate = float(wrong_pivot[('control', negative_condition)].mean())
    gates = {
        'adequate_pairs': bool(len(margin_pivot) >= MINIMUM_ACTIVE_PAIRS),
        'error_shift_exceeds_sesoi': bool(
            results['error_exact_shift']['ci_low'] > MINIMUM_STAGE_EFFECT
        ),
        'error_exact_beats_random': bool(
            results['error_exact_minus_random']['ci_low'] > 0
        ),
        'error_exact_beats_reverse': bool(
            results['error_exact_minus_reverse']['ci_low'] > 0
        ),
        'reverse_induces_expected_error': bool(
            results['control_induction_vs_random']['ci_low'] > 0
        ),
        'reciprocal_score_positive': bool(
            results['reciprocal_score']['ci_low'] > 0
        ),
    }
    stage_decisions[stage] = {
        'passes': bool(all(gates.values())), 'gates': gates,
        'error_rescue_rate': error_rescue_rate,
        'control_induction_rate': induction_rate,
    }

if stage_decisions['net_torque_strength']['passes']:
    PRIMARY_DECISION = 'net_torque_strength_fracture_confirmed'
elif stage_decisions['physical_semantic_handoff']['passes']:
    PRIMARY_DECISION = 'net_torque_to_physical_semantic_handoff_fracture'
elif stage_decisions['response_code_handoff']['passes']:
    PRIMARY_DECISION = 'physical_semantic_to_response_code_fracture'
elif any(
    value['gates']['error_exact_beats_random']
    for value in stage_decisions.values()
):
    PRIMARY_DECISION = 'one_directional_repair_without_reciprocal_confirmation'
else:
    PRIMARY_DECISION = 'fracture_not_localized_at_frozen_sites'

test_frame = pd.DataFrame(test_rows)
test_frame.to_csv(ANALYSIS_DIR / 'reciprocal_bootstrap_tests.csv', index=False)
decision = {
    'schema_version': 'qwen35b-nettorque-reciprocal-decision-1.0',
    'primary_decision': PRIMARY_DECISION,
    'active_pairs': int(len(margin_pivot)),
    'stage_decisions': stage_decisions,
    'claim_boundary': (
        'A passing stage is a reciprocal causal result for this frozen '
        'net-torque-to-outcome error class, not a universal model mechanism.'
    ),
}
write_json_atomic(ANALYSIS_DIR / 'reciprocal_decision.json', decision)
display(test_frame.round(5))
print('Primary decision:', PRIMARY_DECISION)
print(json.dumps(stage_decisions, indent=2))


In [ ]:
%%cell_progress 16/17 Audit matching, leakage, mappings, and norms
cell_stage('Checking the full reciprocal causal claim boundary')

pair_frame = pd.DataFrame(active_pair_manifest)
assert pair_frame['pair_id'].is_unique
assert float(pair_frame['match_distance'].max()) <= MAX_MATCH_DISTANCE
assert set(pair_frame['error_scene_id']).isdisjoint(
    set(pair_frame['control_scene_id'])
)
assert all(
    physical_by_scene[row['error_scene_id']]['outcome']
    == physical_by_scene[row['control_scene_id']]['outcome']
    == row['exact_semantic']
    for row in active_pair_manifest
)
assert all(
    physical_by_scene[row['error_scene_id']]['diagnostic_cell']
    == physical_by_scene[row['control_scene_id']]['diagnostic_cell']
    == row['diagnostic_cell']
    for row in active_pair_manifest
)
assert all(
    str(behavior_by_scene.loc[row['error_scene_id'], 'split']) == 'probe_test'
    and str(behavior_by_scene.loc[row['control_scene_id'], 'split']) == 'probe_test'
    for row in active_pair_manifest
)
assert replay_frame.groupby(['pair_id', 'role'])['stable'].all().all()
assert record_frame.groupby([
    'pair_id', 'role', 'mapping_variant', 'condition', 'alpha'
]).size().eq(1).all()
assert set(record_frame['mapping_variant']) == set(MAPPING_VARIANTS)
assert set(record_frame['condition']) == set(CONDITIONS)
assert set(record_frame['alpha'].astype(float)) == set(ALPHAS)
assert record_frame['patched_tokens'].eq(1).all()

norm_rows = []
for keys, current in record_frame.groupby(
    ['pair_id', 'role', 'mapping_variant', 'alpha']
):
    for prefix in ('l20_', 'l30_', 'l34_'):
        values = current[current['condition'].str.startswith(prefix)][
            'payload_norm'
        ].to_numpy(dtype=float)
        relative = float((values.max() - values.min()) / values.mean())
        norm_rows.append({
            'pair_id': keys[0], 'role': keys[1],
            'mapping_variant': keys[2], 'alpha': keys[3],
            'stage_prefix': prefix, 'relative_range': relative,
        })
norm_frame = pd.DataFrame(norm_rows)
assert float(norm_frame['relative_range'].max()) < NORM_RELATIVE_TOLERANCE
norm_frame.to_csv(ANALYSIS_DIR / 'norm_audit.csv', index=False)

audit_document = {
    'schema_version': 'qwen35b-nettorque-reciprocal-audit-1.0',
    'preexisting_candidate_errors': int(len(candidate_rows)),
    'strict_pairs': int(len(pair_manifest)),
    'mapping_stable_active_pairs': int(len(active_pair_manifest)),
    'maximum_match_distance': float(pair_frame['match_distance'].max()),
    'maximum_self_replay_error': float(max(self_errors.values())),
    'maximum_norm_relative_range': float(norm_frame['relative_range'].max()),
    'selection_uses_current_interventions': False,
    'directions_use_probe_train_dev_or_independent_support': True,
    'targets_are_probe_test_only': True,
    'two_swapped_answer_mappings': True,
    'actual_accelerator': GPU_NAME,
}
write_json_atomic(ANALYSIS_DIR / 'causal_audit.json', audit_document)
display(audit_document)


In [ ]:
%%cell_progress 17/17 Save the reciprocal decision, figure, and report
cell_stage('Writing paper-facing causal artifacts')
import matplotlib.pyplot as plt

stage_order = list(stage_specs)
stage_labels = {
    'net_torque_strength': 'L20\nnet torque',
    'physical_semantic_handoff': 'L30\nphysical side',
    'response_code_handoff': 'L34\nresponse code',
}
plot_rows = []
for stage in stage_order:
    for metric in ('error_exact_minus_random', 'control_induction_vs_random'):
        row = test_frame[
            (test_frame['stage'] == stage) & (test_frame['metric'] == metric)
        ].iloc[0]
        plot_rows.append({'stage': stage, 'metric': metric, **row.to_dict()})
plot_frame = pd.DataFrame(plot_rows)

fig, axis = plt.subplots(figsize=(10.5, 5.2))
x = np.arange(len(stage_order))
width = 0.34
for index, (metric, label, color) in enumerate((
    ('error_exact_minus_random', 'Repair error: exact − random', '#2563EB'),
    ('control_induction_vs_random', 'Induce error: random − reversed', '#D97706'),
)):
    current = plot_frame[plot_frame['metric'] == metric].set_index(
        'stage'
    ).reindex(stage_order)
    y = current['estimate'].to_numpy(dtype=float)
    axis.bar(
        x + (index - 0.5) * width, y, width,
        label=label, color=color,
        yerr=np.vstack([
            np.maximum(0, y - current['ci_low'].to_numpy(dtype=float)),
            np.maximum(0, current['ci_high'].to_numpy(dtype=float) - y),
        ]), capsize=4,
    )
axis.axhline(0, color='black', linewidth=0.8)
axis.set_xticks(x, [stage_labels[stage] for stage in stage_order])
axis.set_ylabel('Correct-direction log-odds advantage')
axis.set_title('Qwen35B reciprocal net-torque fracture test')
axis.legend(frameon=False)
fig.tight_layout()
figure_path = FIGURE_DIR / 'qwen35b_reciprocal_nettorque_causal.png'
fig.savefig(figure_path, dpi=220, bbox_inches='tight')
plt.show()

def interval(stage, metric):
    row = test_frame[
        (test_frame['stage'] == stage) & (test_frame['metric'] == metric)
    ].iloc[0]
    return f"{row['estimate']:.4f} [{row['ci_low']:.4f}, {row['ci_high']:.4f}]"

lines = [
    '# Qwen35B Net-Torque Reciprocal Causal Test', '',
    f'**Primary decision:** {PRIMARY_DECISION}', '',
    '## Frozen support', '',
    f'- Preexisting net-torque-to-outcome errors: {len(candidate_rows)}',
    f'- Strict behavior/physics pairs: {len(pair_manifest)}',
    f'- Mapping-stable reciprocal pairs: {len(active_pair_manifest)}',
    f'- Maximum retained physical match distance: {pair_frame["match_distance"].max():.3f}',
    '', '## Reciprocal causal effects at alpha = 1', '',
]
for stage in stage_order:
    lines.extend([
        f'### {stage_labels[stage].replace(chr(10), " ")}', '',
        f"- Error repair versus random: {interval(stage, 'error_exact_minus_random')}",
        f"- Correct-scene induction versus random: {interval(stage, 'control_induction_vs_random')}",
        f"- Reciprocal score: {interval(stage, 'reciprocal_score')}",
        f"- Gate passed: {stage_decisions[stage]['passes']}", '',
    ])
lines.extend([
    '## Interpretation boundary', '',
    (
        'A passing stage means the frozen correct-direction intervention both '
        'repairs this preexisting error class and the reversed intervention '
        'induces the predicted error in strictly matched correct scenes. It '
        'does not prove a pure one-dimensional variable or a universal route '
        'for every model and physical task.'
    ), '',
])
report_path = ANALYSIS_DIR / 'RECIPROCAL_CAUSAL_REPORT.md'
report_path.write_text('\n'.join(lines), encoding='utf-8')

run_complete = {
    'schema_version': 'qwen35b-nettorque-reciprocal-complete-1.0',
    'status': 'complete', 'model_id': MODEL_ID, 'revision': MODEL_REVISION,
    'precision': MODEL_DTYPE, 'primary_decision': PRIMARY_DECISION,
    'candidate_errors': len(candidate_rows),
    'strict_pairs': len(pair_manifest),
    'active_pairs': len(active_pair_manifest),
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'active_lock_sha256': sha256_file(ACTIVE_LOCK_PATH),
    'decision_sha256': sha256_file(ANALYSIS_DIR / 'reciprocal_decision.json'),
    'figure': str(figure_path), 'report': str(report_path),
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
}
write_json_atomic(OUTPUT_ROOT / 'RUN_COMPLETE.json', run_complete)
print(report_path.read_text(encoding='utf-8'))
print('RUN_COMPLETE:', OUTPUT_ROOT / 'RUN_COMPLETE.json')
